# LogAI - Log Preprocessing

## Objective

This notebook reads raw telecom logs, parses them into structured format,
extracts important features, performs preprocessing, and saves the processed
dataset for downstream machine learning models.

### Input
- Raw Log File (.log)

### Output
- processed_logs.csv
- parsed_logs.parquet
- feature_summary.csv

In [5]:

import pandas as pd
import numpy as np
import re
import warnings
from pathlib import Path
from tqdm import tqdm

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)

Defining Project Paths

In [6]:
# Project Root

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data"

RAW_LOG_DIR = DATA_DIR / "raw_logs"

PROCESSED_DIR = DATA_DIR / "processed"

EMBEDDING_DIR = DATA_DIR / "embeddings"

print(PROJECT_ROOT)

..


Reading log files

In [7]:
LOG_FILE = RAW_LOG_DIR / "Sample data.log"

with open(LOG_FILE, "r", encoding="utf-8", errors="ignore") as file:
    logs = file.readlines()

print("=" * 50)
print("Total Logs :", len(logs))
print("=" * 50)

Total Logs : 5000


Quick Log Inspection

In [8]:
for i in range(10):
    print(logs[i])

[2026-05-16 03:00:26.877] [ERROR] [Router] user=1683 ip=10.149.39.145 session=S106183 code=459 msg="Database timeout"

[2026-05-16 02:46:57.932] [WARN] [Router] user=5877 ip=10.241.14.234 session=S886684 code=411 msg="Slow database response"

[2026-05-16 02:54:21.917] [WARN] [Auth] user=1374 ip=10.203.62.140 session=S646470 code=238 msg="Retry initiated"

[2026-05-16 02:54:40.319] [ERROR] [Firewall] user=1612 ip=10.66.220.16 session=S889455 code=344 msg="Connection refused"

[2026-05-16 02:48:09.842] [INFO] [DB] user=7131 ip=10.222.244.207 session=S635519 code=491 msg="Cache refreshed"

[2026-05-16 02:53:01.552] [INFO] [Firewall] user=4559 ip=10.61.207.247 session=S156355 code=332 msg="Heartbeat received"

[2026-05-16 02:54:35.089] [INFO] [Firewall] user=1920 ip=10.237.223.115 session=S297242 code=335 msg="Message delivered"

[2026-05-16 02:47:00.310] [WARN] [Router] user=2913 ip=10.144.77.157 session=S428019 code=207 msg="Queue length increasing"

[2026-05-16 02:53:52.449] [ERROR] [SM

Empty Record List

In [9]:
records = []

Log Parser Function

In [10]:
import re

def parse_log(log):

    record = {}

    log = log.strip()

    record["Raw_Log"] = log

    # -------------------------------------------------
    # Basic Metadata
    # -------------------------------------------------

    component = re.search(r'\[(.*?)\|', log)
    line_number = re.search(r'\|(\d{5})\|', log)
    timestamp = re.search(r'(\d{2}:\d{2}:\d{2}:\d{3})', log)
    severity = re.search(r'(~CR~|~ER~|~WR~)', log)

    record["Component"] = component.group(1) if component else None
    record["Line_Number"] = line_number.group(1) if line_number else None
    record["Timestamp"] = timestamp.group(1) if timestamp else None
    record["Severity"] = severity.group(1) if severity else None

    # -------------------------------------------------
    # User Information
    # -------------------------------------------------

    username = re.search(r'username:([^,\]]+)', log)

    account = re.search(r'AccountId:(\d+)', log)

    oa = re.search(r'OA:([^,\]]+)', log)

    da = re.search(r'DA:(\d+)', log)

    record["Username"] = username.group(1) if username else None
    record["AccountID"] = account.group(1) if account else None
    record["Originating_Address"] = oa.group(1) if oa else None
    record["Destination_Address"] = da.group(1) if da else None

    # -------------------------------------------------
    # Performance Metrics
    # -------------------------------------------------

    tps = re.search(r'TPS:\s*(\d+)', log)

    queue = re.search(r'Count:(\d+)', log)

    packet = re.search(r'SPKTh No.:(\d+)', log)

    record["TPS"] = tps.group(1) if tps else None
    record["Queue_Count"] = queue.group(1) if queue else None
    record["Packet_Number"] = packet.group(1) if packet else None

    # -------------------------------------------------
    # Boolean Flags
    # -------------------------------------------------

    record["AuthenticationSuccess"] = int(
        "Authentication Success" in log
    )

    record["PositiveResponse"] = int(
        "Positive SMSC Resp" in log
    )

    record["QueueInserted"] = int(
        "Inserted SMPP Packet" in log
    )

    record["RouterSent"] = int(
        "Sent Byte to Router" in log
    )

    record["ContainsError"] = int(
        any(x in log.upper() for x in [
            "ERROR",
            "FAILED",
            "TIMEOUT",
            "NOT FOUND"
        ])
    )

    # -------------------------------------------------
    # Text Features
    # -------------------------------------------------

    record["Message_Length"] = len(log)

    record["Word_Count"] = len(log.split())

    return record

In [11]:
records = []

for log in tqdm(logs):
    records.append(parse_log(log))

df = pd.DataFrame(records)

df.head()

100%|██████████| 5000/5000 [00:00<00:00, 113934.17it/s]


,Raw_Log,Component,Line_Number,Timestamp,Severity,Username,AccountID,Originating_Address,Destination_Address,TPS,Queue_Count,Packet_Number,AuthenticationSuccess,PositiveResponse,QueueInserted,RouterSent,ContainsError,Message_Length,Word_Count
0,"[2026-05-16 03:00:26.877] [ERROR] [Router] user=1683 ip=10.149.39.145 session=S106183 code=459 msg=""Database timeout""",None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,1,117,10
1,"[2026-05-16 02:46:57.932] [WARN] [Router] user=5877 ip=10.241.14.234 session=S886684 code=411 msg=""Slow database res...",None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,122,11
2,"[2026-05-16 02:54:21.917] [WARN] [Auth] user=1374 ip=10.203.62.140 session=S646470 code=238 msg=""Retry initiated""",None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,113,10
3,"[2026-05-16 02:54:40.319] [ERROR] [Firewall] user=1612 ip=10.66.220.16 session=S889455 code=344 msg=""Connection refu...",None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,1,120,10
4,"[2026-05-16 02:48:09.842] [INFO] [DB] user=7131 ip=10.222.244.207 session=S635519 code=491 msg=""Cache refreshed""",None,None,None,None,None,None,None,None,None,None,None,0,0,0,0,0,112,10


In [12]:
df["Timestamp"] = pd.to_datetime(
    df["Timestamp"],
    format="%H:%M:%S:%f",
    errors="coerce"
)

numeric_cols = [
    "Line_Number",
    "AccountID",
    "TPS",
    "Queue_Count",
    "Packet_Number"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [13]:
df["Hour"] = df["Timestamp"].dt.hour
df["Minute"] = df["Timestamp"].dt.minute
df["Second"] = df["Timestamp"].dt.second
df["Millisecond"] = (
    df["Timestamp"].dt.microsecond // 1000
)

In [14]:
for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].fillna("Unknown")

    else:

        df[col] = df[col].fillna(0)

In [15]:
print("Before :", len(df))

df = df.drop_duplicates()

print("After :", len(df))

Before : 5000
After : 5000


In [16]:
summary = pd.DataFrame({

    "Missing Values": df.isnull().sum(),

    "Unique Values": df.nunique(),

    "Data Type": df.dtypes.astype(str)

})

summary

,Missing Values,Unique Values,Data Type
Raw_Log,0,5000,str
Component,0,1,object
Line_Number,0,1,float64
Timestamp,0,1,object
Severity,0,1,object
Username,0,1,object
AccountID,0,1,float64
Originating_Address,0,1,object
Destination_Address,0,1,object
TPS,0,1,float64


In [17]:
# Convert all object columns to string

object_cols = df.select_dtypes(include=["object"]).columns

for col in object_cols:
    df[col] = df[col].astype(str)

print("Converted object columns to string.")

Converted object columns to string.


In [19]:
df.to_parquet(
    processed_path / "parsed_logs.parquet",
    index=False
)

NameError: name 'processed_path' is not defined

In [ ]:
import os

os.listdir("../data/processed")

['parsed_logs.parquet', 'processed_logs.csv']

In [ ]:
from pathlib import Path

for file in Path("../data/processed").iterdir():
    print(file.name)

parsed_logs.parquet
processed_logs.csv


In [ ]:
from pathlib import Path

processed_path = Path("../data/processed")

processed_path.mkdir(parents=True, exist_ok=True)

df.to_csv(
    processed_path / "processed_logs.csv",
    index=False
)

df.to_parquet(
    processed_path / "parsed_logs.parquet",
    index=False
)

summary.to_csv(
    processed_path / "feature_summary.csv"
)

print("Files Saved Successfully")

✅ Files Saved Successfully
